## Users and Groups
A user has one **primary group** and can belong to multiple **secondary (supplementary) groups**. Group information is stored in the `/etc/group` file.
 
To find which groups a user belongs to, use the `groups` command: 
```sh
$ groups stevejobs
stevejobs : stevejobs adm dialout cdrom floppy sudo audio dip video plugdev netdev
 
$ groups root
root : root
```

Users are added to or removed from groups by editing `/etc/group`. A typical file looks like this: 
```
root:x:0:
adm:x:4:syslog,stevejobs
stevejobs:x:1000:
```
 
Each line follows the format:
```
group_name:password:GID:member_list
```
 
1. Group name: human-readable name of the group
2. Password: group password placeholder which is almost always `x` today (the real hash, if any, lives in `/etc/gshadow`)
3. GID: numeric group ID, used internally by the kernel
4. Member list: comma-separated usernames belonging to this group as a secondary group
 
> **Note:** A user's *primary* group is defined separately, in `/etc/passwd`, not listed in the member field of `/etc/group`.
 
To add a user `billgates` to the `adm` group, append the username to that group's member list:
```
root:x:0:
adm:x:4:syslog,stevejobs,billgates
stevejobs:x:1000:
```
 
> **Important:**
> - `/etc/group` can only be modified by the root user.
> - Changes take effect only after the affected user logs out and logs back in.

## File and Directory Permissions
Running `ls -l` shows a descriptive representation of every file's permissions, ownership, and type:  
![ls long](https://i.imgur.com/wJT1uqP.png)
 
**Ownership:** whoever creates a file becomes its owner. In the example above, `.bash_history` is owned by the user `stevejobs` and belongs to the `stevejobs` group. Ownership is closely tied to how permissions are interpreted.
 
**Permissions:** each file/directory has three permission types: 
- `r` is read
- `w` is write
- `x` is execute
 
By default, a file created by a regular user gets `rw-rw-r--` permissions (this default can be changed with the `umask` command).
 
These permissions mean different things depending on whether they apply to a file or a directory. For files:
- Read means view the contents of the file
- Write means update or delete the file
- Execute means run the file as a program/script

For directories:
- Read means list the contents of the directory
- Write means create, rename, or delete files within the directory, and modify the directory's own attributes
- Execute means `cd` into the directory and access the files/subdirectories inside it

### Changing Permissions and Ownership
Use `chmod` command to change permissions. Permissions for owner, group and other are represented numerically:
- 4 means read
- 2 means write
- 1 means execute

Therfore,
- 7 = 4+2+1 means read write execute
- 5 = 4+1 means read and execute

**Example:** Set owner to read/write/execute, and group + other to read/execute: 
```sh
$ chmod 755 myfile.txt
```
 
### Change Ownership
To change ownership of file use `chown` command:
```sh
# Change user ownership of a file
$ chown billgates myfile.txt
 
# Change group ownership of a file
$ chown :microsoft myfile.txt
 
# Change both user and group ownership of a file
$ chown billgates:microsoft myfile.txt
```
 
> **Note:** Only the file/directory owner and root can run `chmod` and `chown`.

### suid and sgid
When you're logged in, the commands you run act with your own permissions. But some tasks like changing your password need to modify a file owned by root (`/etc/shadow`), which you normally can't touch.

Linux solves this with two special permissions: `suid` (set user id) and `sgid` (set group id).
- If a program has `suid` set, it runs as the file's owner, not as the user who started it.
- If a program has `sgid` set, it runs as the file's group, not the user's own group.

A program can have one, both, or neither of these set.
```sh
$ ls -l /usr/bin/passwd
-rwsr-xr-x 1 root root 68208 May 28  2020 /usr/bin/passwd
```

Notice the `s` where you'd normally see `x`. This means `suid` is set. Since `passwd` is owned by root, any regular user running this command actually runs it as root, which is exactly how a normal user is able to update the root-owned `/etc/shadow` file when changing their password.

### Sticky Bit
Anyone with write permission to a directory can delete files in it. This might be acceptable for a group project, but is not desirable for globally shared file space such as the `/tmp` directory. Multiple programs write files to the `/tmp` directory but one program wouldn't want other program to delete its files. We can set sticky bit for a directory and only the root user or the owner can delete files in that directory.

```sh
$ chmod +t test
$ ls -ld test
drwxr-xr-t 2 stevejobs stevejobs 4096 Dec  4 20:50 test
```

Small t means sticky bit + executable. Capital T means sticky bit without execute permission. Sticky bit has no meaning for files and is ignored.